# Line vs Rectangle Convergence (Paper Output Only)

This cleaned notebook keeps only the code required to generate:

- `best error for different Ns.pdf` (Figure 4)

It supports two modes:

1. Load precomputed arrays (`l2norm_rw.npy`, `l2norm_delve.npy`) if available.
2. Recompute them (optional, expensive).


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'


In [ ]:
def comp_sig(n: int, d: int, C: float) -> float:
    return C * ((np.log(n) / n) ** (1 / ((d / 2) + 3)))


def kernel_matrix(X: np.ndarray, epsilon: float) -> np.ndarray:
    D = squareform(pdist(X, metric='euclidean'))
    return np.exp(-(D ** 2) / (epsilon ** 2))


def laplacian_rw(W: np.ndarray):
    D_inv = np.diag(1 / np.sum(W, axis=1))
    L_rw = np.eye(W.shape[0]) - D_inv @ W

    d, v = np.linalg.eig(L_rw)
    idx = np.argsort(d.real)
    return L_rw, d[idx].real, v[:, idx].real


def laplacian_sym(W: np.ndarray):
    D_half_inv = np.diag(np.sum(W, axis=1) ** (-0.5))
    L_sym = D_half_inv @ W @ D_half_inv

    d, v = np.linalg.eigh(L_sym)
    idx = np.argsort(d)[::-1]
    return L_sym, d[idx], v[:, idx]


def calc_differential_vec(L_A: np.ndarray, v_B: np.ndarray, k: int):
    U1 = v_B[:, :k]
    Q1 = np.eye(U1.shape[0]) - U1 @ U1.T
    Q1 = Q1 @ L_A @ Q1

    s, u1 = np.linalg.eigh(Q1)
    idx = np.argsort(s)[::-1]
    return s[idx], u1[:, idx]


def rectangle_converge(L: float, W: float, N: int, C_vec: np.ndarray, k: int = 5):
    w = np.random.uniform(0, W, N)
    l = np.random.uniform(0, L, N)

    X1 = l[:, np.newaxis]
    X2 = np.column_stack((l, w))

    v_th = np.cos(np.pi * w / W)
    v_th = v_th / np.linalg.norm(v_th)

    v_thl = np.cos(np.pi * l / L)
    v_thl = v_thl / np.linalg.norm(v_thl)

    rw_errors = []
    delve_errors = []

    for C in C_vec:
        eps1 = comp_sig(N, 1, C)
        eps2 = comp_sig(N, 2, C)

        K1 = kernel_matrix(X1, eps1)
        K2 = kernel_matrix(X2, eps2)

        _, _, v1_rw = laplacian_rw(K1)
        _, _, v1_sym = laplacian_sym(K1)
        L2_sym, _, _ = laplacian_sym(K2)

        _, u1 = calc_differential_vec(L2_sym, v1_sym, k)

        alpha_l = np.inner(v_thl, v1_rw[:, 1])
        alpha_w = np.inner(v_th, u1[:, 0])

        rw_errors.append(np.linalg.norm(v1_rw[:, 1] - alpha_l * v_thl))
        delve_errors.append(np.linalg.norm(u1[:, 0] - alpha_w * v_th) ** 2)

    return np.array(rw_errors), np.array(delve_errors)


In [ ]:
# Shared grid from original notebook
N_list = np.round(10 ** np.array([3, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7])).astype(int)
C_vec = np.arange(0.6, 2.0, 0.2)
B = len(N_list)
C = len(C_vec)


In [ ]:
# Prefer loading precomputed arrays (fast); recompute only if needed
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'notebooks').exists() and (PROJECT_ROOT.name == 'notebooks'):
    PROJECT_ROOT = PROJECT_ROOT.parent

candidates = [
    PROJECT_ROOT,
    Path('/Users/shiraalon/Documents/Thesis/Different examples for FKT comparation'),
]

rw_path = None
delve_path = None
for base in candidates:
    p1 = base / 'l2norm_rw.npy'
    p2 = base / 'l2norm_delve.npy'
    if p1.exists() and p2.exists():
        rw_path, delve_path = p1, p2
        break

RUN_SIMULATION = rw_path is None

if RUN_SIMULATION:
    print('Precomputed arrays not found; running simulation (this may take a while).')
    R = 100
    rw_l_convergence = np.zeros((R, B, C))
    delve_w_convergence = np.zeros((R, B, C))

    for r in range(R):
        print(f'Run {r + 1}/{R}')
        for i, N in enumerate(N_list):
            rw_l_convergence[r, i, :], delve_w_convergence[r, i, :] = rectangle_converge(
                L=4,
                W=2,
                N=N,
                C_vec=C_vec,
                k=5,
            )

    np.save(PROJECT_ROOT / 'l2norm_rw.npy', rw_l_convergence)
    np.save(PROJECT_ROOT / 'l2norm_delve.npy', delve_w_convergence)
else:
    print(f'Loading precomputed arrays from:
  {rw_path}
  {delve_path}')
    rw_l_convergence = np.load(rw_path)
    delve_w_convergence = np.load(delve_path)

print('rw_l_convergence shape:', rw_l_convergence.shape)
print('delve_w_convergence shape:', delve_w_convergence.shape)


In [ ]:
# Aggregate over repetitions and pick best C per N
rw_l_mean = rw_l_convergence.mean(axis=0)
rw_l_std = rw_l_convergence.std(axis=0)

d_w_mean = delve_w_convergence.mean(axis=0)
d_w_std = delve_w_convergence.std(axis=0)

rw_opt = rw_l_mean.min(axis=1)
delve_opt = d_w_mean.min(axis=1)

rw_opt_std = rw_l_std[np.arange(B), rw_l_mean.argmin(axis=1)]
delve_opt_std = d_w_std[np.arange(B), d_w_mean.argmin(axis=1)]


In [ ]:
# Output: best error for different Ns.pdf
plt.figure(figsize=(6, 4))

plt.loglog(N_list, rw_opt, 'o-', label='Single manifold')
plt.loglog(N_list, np.sqrt(delve_opt), 'o-', label='DELVE')

plt.xlabel('n', fontsize=18)
plt.ylabel('Optimal error over bandwidth', fontsize=18)
plt.legend(fontsize=15)
plt.grid(alpha=0.3, which='both')

# Keep the fixed slope annotation used in the original paper figure
textstr = '
'.join((
    r'Slope: single manifold = -0.4133',
    r'Slope: DELVE = -0.216',
))

plt.text(
    0.19 * max(N_list),
    0.53 * max(rw_opt),
    textstr,
    fontsize=13,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
)

plt.tick_params(axis='both', which='major', labelsize=15)
plt.tick_params(axis='both', which='minor', labelsize=15)
plt.tight_layout()
plt.savefig('best error for different Ns.pdf')
plt.show()
